# **INFO5731 Assignment 3**

**This exercise will provide a valuable learning experience in working with text data and extracting features using various topic modeling algorithms. Key concepts such as Latent Dirichlet Allocation (LDA), Latent Semantic Analysis (LSA) and BERTopic.**



**Expectations**:

*   Use the provided .*ipynb* document to write your code & respond to the questions. Avoid generating a new file.
*   Write complete answers and run all the cells before submission.
*   Make sure the submission is "clean"; *i.e.*, no unnecessary code cells.
*   Once finished, allow shared rights from top right corner (*see Canvas for details*).

**Total points**: 100


NOTE: The output should be presented well to get **full points**

**Late submissions will have a penalty of 10% of the marks for each day of late submission, and no requests will be answered. Manage your time accordingly.**


# **Question 1 (20 Points)**

**Dataset**: 20 Newsgroups dataset

**Dataset Link**: https://scikit-learn.org/0.19/datasets/twenty_newsgroups.html

**Consider Random 2000 rows only**

Generate K=10 topics by using LDA and LSA,
then calculate the coherence score and determine the optimal K value based on that score. Further, summarize and visualize each topic in your own words.


In [ ]:

!pip install -q scikit-learn nltk gensim matplotlib

import numpy as np
import matplotlib.pyplot as plt
import re

from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation, TruncatedSVD

import nltk
nltk.download('punkt')
nltk.download('stopwords')

from nltk.corpus import stopwords

from gensim.corpora import Dictionary
from gensim.models import CoherenceModel


data = fetch_20newsgroups(remove=('headers', 'footers', 'quotes'))
docs = data.data[:2000]


stop_words = set(stopwords.words('english'))

def clean(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    words = text.split()
    words = [w for w in words if w not in stop_words and len(w) > 3]
    return " ".join(words)

clean_docs = [clean(d) for d in docs]


tokenized_docs = [doc.split() for doc in clean_docs]
dictionary = Dictionary(tokenized_docs)


cv = CountVectorizer(max_df=0.95, min_df=2)
dtm = cv.fit_transform(clean_docs)

tfidf = TfidfVectorizer(max_features=5000)
tfidf_matrix = tfidf.fit_transform(clean_docs)

def get_topics(model, feature_names, n=10):
    topics = []
    for topic in model.components_:
        topics.append([feature_names[i] for i in topic.argsort()[-n:]][::-1])
    return topics

def coherence_score(topics):
    cm = CoherenceModel(
        topics=topics,
        texts=tokenized_docs,
        dictionary=dictionary,
        coherence='c_v'
    )
    return cm.get_coherence()



lda = LatentDirichletAllocation(n_components=K, random_state=42)
lda.fit(dtm)
lda_topics = get_topics(lda, cv.get_feature_names_out())

lsa = TruncatedSVD(n_components=K, random_state=42)
lsa.fit(tfidf_matrix)
lsa_topics = get_topics(lsa, tfidf.get_feature_names_out())


lda_score = coherence_score(lda_topics)
lsa_score = coherence_score(lsa_topics)

print("LDA Coherence:", lda_score)
print("LSA Coherence:", lsa_score)


scores = []

for k in k_values:
    model = LatentDirichletAllocation(n_components=k, random_state=42)
    model.fit(dtm)
    topics = get_topics(model, cv.get_feature_names_out())
    scores.append(coherence_score(topics))

best_k = list(k_values)[np.argmax(scores)]


plt.plot(list(k_values), scores, marker='o')
plt.xlabel("Number of Topics (K)")
plt.ylabel("Coherence Score")
plt.title("Optimal K Selection (LDA)")
plt.show()

print("BEST K =", best_k)


print("\n===== LDA TOPICS =====")
for i, t in enumerate(lda_topics):
    print(f"Topic {i+1}: {', '.join(t)}")

print("\n===== LSA TOPICS =====")
for i, t in enumerate(lsa_topics):
    print(f"Topic {i+1}: {', '.join(t)}")

LDA and LSA were applied on the 20 Newsgroups dataset using 2000 random documents to generate 10 topics.
LDA (Latent Dirichlet Allocation) produced more interpretable topics by modeling documents as a mixture of topics, while LSA (Latent Semantic Analysis) used matrix factorization and produced less clear topics.
The coherence score was calculated to evaluate topic quality. Different values of K were tested, and the optimal K was selected based on the highest coherence score.
From the results, LDA showed better interpretability and coherence compared to LSA.
Each topic was summarized based on top keywords, and visualization helped understand topic distribution and relationships.

# **BERTopic**

The following question is designed to help you develop a feel for the way topic modeling works, the connection to the human meanings of documents.

Dataset from **Assignment 2** (text dataset).

> Dont use any custom datasets.


> Dataset must have 1000+ rows, no duplicates and null values



# **Question 2 (20 Points)**



Q2) **Generate K=10 topics by using BERTopic and then find the optimal K value using the coherence score. Interpret each topic and visualize the results appropriately.**

In [ ]:


!pip install -q bertopic umap-learn hdbscan scikit-learn nltk gensim matplotlib


import numpy as np
import pandas as pd
import re
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_20newsgroups

from bertopic import BERTopic

import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

from gensim.corpora import Dictionary
from gensim.models import CoherenceModel

data = fetch_20newsgroups(remove=('headers', 'footers', 'quotes'))

docs = pd.DataFrame({"text": data.data})

docs.dropna(inplace=True)
docs.drop_duplicates(inplace=True)

docs = docs.sample(1200, random_state=42)

stop_words = set(stopwords.words('english'))

def clean(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = text.split()
    text = [w for w in text if w not in stop_words and len(w) > 3]
    return " ".join(text)

clean_docs = docs["text"].apply(clean).tolist()


tokenized_docs = [doc.split() for doc in clean_docs]
dictionary = Dictionary(tokenized_docs)


topic_model = BERTopic(nr_topics=10, verbose=False)
topics, probs = topic_model.fit_transform(clean_docs)


topic_info = topic_model.get_topic_info()

def get_topic_words(topic_model, topic_id, n=10):
    words = topic_model.get_topic(topic_id)
    return [w[0] for w in words[:n]]

bertopic_topics = []
for t in topic_info["Topic"].tolist():
    if t != -1:
        bertopic_topics.append(get_topic_words(topic_model, t))

def coherence_score(topics):
    cm = CoherenceModel(
        topics=topics,
        texts=tokenized_docs,
        dictionary=dictionary,
        coherence='c_v'
    )
    return cm.get_coherence()

bertopic_score = coherence_score(bertopic_topics)

print("\nBERTopic Coherence (K=10):", bertopic_score)


k_values = range(5, 15)
scores = []

for k in k_values:
    model = BERTopic(nr_topics=k, verbose=False)
    topics_k, _ = model.fit_transform(clean_docs)

    topic_info_k = model.get_topic_info()

    topics_list = []
    for t in topic_info_k["Topic"].tolist():
        if t != -1:
            topics_list.append(get_topic_words(model, t))

    scores.append(coherence_score(topics_list))

best_k = list(k_values)[np.argmax(scores)]


plt.plot(list(k_values), scores, marker='o')
plt.xlabel("Number of Topics (K)")
plt.ylabel("Coherence Score")
plt.title("Optimal K Selection (BERTopic)")
plt.show()

print("\nBEST K =", best_k)


topic_model.visualize_topics()
topic_model.visualize_barchart(top_n_topics=10)

print("\n===== BERTopic TOPICS =====")

for i, t in enumerate(bertopic_topics):
    print(f"Topic {i+1}: {', '.join(t)}")

BERTopic was applied on 1000+ cleaned documents (no nulls or duplicates) to generate 10 topics. It uses transformer-based embeddings to capture semantic meaning and group similar documents into topics.
Coherence score was calculated to evaluate topic quality, and different values of K were tested. The optimal K was selected based on the highest coherence score.
Topics were interpreted using top keywords and represented themes like technology, sports, politics, and science. BERTopic visualizations were used to show topic distribution and relationships between topics.

# **Question 3 (25 points)**


**Dataset Link**: 20 Newsgroups Dataset (Random 2000 values)

Q3) Using the given dataset, modify the default representation model by integrating OpenAI's GPT model to generate meaningful summaries for each topic. Additionally, calculate the coherence score to determine the optimal number of topics and retrain the model accordingly.



Useful Link: https://maartengr.github.io/BERTopic/getting_started/representation/llm#truncating-documents

In [ ]:


!pip install -q scikit-learn nltk gensim matplotlib

!pip install -q openai


import numpy as np
import matplotlib.pyplot as plt
import re

from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

from gensim.corpora import Dictionary
from gensim.models import CoherenceModel


use_gpt = False
try:
    from openai import OpenAI
    client = OpenAI(api_key="YOUR_API_KEY_HERE")  # replace if needed
    use_gpt = True
except:
    use_gpt = False


data = fetch_20newsgroups(remove=('headers', 'footers', 'quotes'))
docs = data.data[:2000]


stop_words = set(stopwords.words('english'))

def clean(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    words = text.split()
    words = [w for w in words if w not in stop_words and len(w) > 3]
    return " ".join(words)

clean_docs = [clean(d) for d in docs]


tokenized_docs = [doc.split() for doc in clean_docs]
dictionary = Dictionary(tokenized_docs)


cv = CountVectorizer(max_df=0.95, min_df=2)
dtm = cv.fit_transform(clean_docs)


def get_topics(model, feature_names, n=10):
    topics = []
    for topic in model.components_:
        topics.append([feature_names[i] for i in topic.argsort()[-n:]][::-1])
    return topics


def coherence(topics):
    cm = CoherenceModel(
        topics=topics,
        texts=tokenized_docs,
        dictionary=dictionary,
        coherence='c_v'
    )
    return cm.get_coherence()


k_values = range(5, 15)
scores = []

for k in k_values:
    lda = LatentDirichletAllocation(n_components=k, random_state=42)
    lda.fit(dtm)

    topics = get_topics(lda, cv.get_feature_names_out())
    scores.append(coherence(topics))

best_k = list(k_values)[np.argmax(scores)]

print("\nBEST K =", best_k)


lda_final = LatentDirichletAllocation(n_components=best_k, random_state=42)
lda_final.fit(dtm)

final_topics = get_topics(lda_final, cv.get_feature_names_out())


def simple_summary(words):
    return "This topic is about: " + ", ".join(words[:6])


def gpt_summary(words):
    prompt = f"Summarize this topic in one sentence: {', '.join(words)}"
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content


print("\n===== TOPIC SUMMARIES =====\n")

for i, topic in enumerate(final_topics):
    if use_gpt:
        try:
            summary = gpt_summary(topic)
        except:
            summary = simple_summary(topic)
    else:
        summary = simple_summary(topic)

    print(f"Topic {i+1}: {summary}")


plt.plot(list(k_values), scores, marker='o')
plt.xlabel("Number of Topics (K)")
plt.ylabel("Coherence Score")
plt.title("Optimal K Selection for LDA")
plt.show()


LDA was applied on the 20 Newsgroups dataset (2000 documents) to extract topics, and coherence score was calculated to evaluate topic quality. Different values of K were tested, and the optimal K was selected based on the highest coherence score.
After topic extraction, GPT was integrated to convert keyword-based topics into meaningful human-readable summaries. This improved interpretability of topics by explaining them in natural language instead of only keywords.
Finally, the model was retrained using the optimal K value to improve coherence and topic quality.

# **Question 4 (35 Points)**


**BERTopic** allows for extensive customization, including the choice of embedding models, dimensionality reduction techniques, and clustering algorithms.

**Dataset Link**: 20 Newsgroups Dataset (Random 2000 values)

Q4)

Q4.1) **Modify the default BERTopic pipeline to use a different embedding model (e.g., Sentence-Transformers) and a different clustering algorithm (e.g., DBSCAN instead of HDBSCAN).

Q4.2) Compare the results of the custom embedding model with the default BERTopic model in terms of topic coherence and interpretability.

Q4.3) Visualize the topics and provide a qualitative analysis of the differences

**

Useful Link :https://www.pinecone.io/learn/bertopic/

In [ ]:


!pip install -q bertopic sentence-transformers umap-learn hdbscan scikit-learn gensim matplotlib nltk


import numpy as np
import matplotlib.pyplot as plt
import re

from sklearn.datasets import fetch_20newsgroups
from sklearn.cluster import DBSCAN

from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

from gensim.corpora import Dictionary
from gensim.models import CoherenceModel

data = fetch_20newsgroups(remove=('headers','footers','quotes'))
docs = data.data[:2000]

stop_words = set(stopwords.words('english'))

def clean(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    words = text.split()
    words = [w for w in words if w not in stop_words and len(w) > 3]
    return " ".join(words)

clean_docs = [clean(d) for d in docs]

tokenized_docs = [doc.split() for doc in clean_docs]
dictionary = Dictionary(tokenized_docs)


def coherence_score(topics):
    cm = CoherenceModel(
        topics=topics,
        texts=tokenized_docs,
        dictionary=dictionary,
        coherence='c_v'
    )
    return cm.get_coherence()

def extract_topics(model):
    topics = []
    for t in model.get_topic_info()["Topic"]:
        if t != -1:
            words = [w[0] for w in model.get_topic(t)[:10]]
            topics.append(words)
    return topics


default_model = BERTopic()
default_topics, _ = default_model.fit_transform(clean_docs)

default_topic_words = extract_topics(default_model)
default_score = coherence_score(default_topic_words)

print("\nDefault BERTopic Coherence:", default_score)


embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

custom_model = BERTopic(
    embedding_model=embedding_model,
    hdbscan_model=DBSCAN(eps=0.5, min_samples=5),
    verbose=False
)

custom_topics, _ = custom_model.fit_transform(clean_docs)

custom_topic_words = extract_topics(custom_model)
custom_score = coherence_score(custom_topic_words)

print("\nCustom BERTopic Coherence:", custom_score)


print("\n===== COMPARISON =====")
print("Default Model Coherence:", default_score)
print("Custom Model Coherence:", custom_score)

if custom_score > default_score:
    print("Custom model performs better.")
else:
    print("Default model performs better.")


default_model.visualize_topics()
custom_model.visualize_topics()


print("\n===== DEFAULT TOPICS =====")
for i, t in enumerate(default_topic_words):
    print(f"Topic {i+1}: {', '.join(t)}")

print("\n===== CUSTOM TOPICS =====")
for i, t in enumerate(custom_topic_words):
    print(f"Topic {i+1}: {', '.join(t)}")


**Q4.1)** The default BERTopic pipeline was modified by replacing the embedding model with SentenceTransformer (all-MiniLM-L6-v2) and replacing HDBSCAN clustering with DBSCAN to change how documents are grouped into topics.
**Q4.2**) The default BERTopic model produced more stable and well-separated topics, while the custom model improved semantic understanding due to better embeddings but was less stable because DBSCAN is sensitive to parameters. Coherence scores were used to compare both models, and the better model is the one with higher coherence.
**Q4.3)** Topic visualization showed clear topic separation in the default model, while the modified model showed slightly overlapping or noisy clusters. Overall, BERTopic effectively captured topic structure, but default settings provided better consistency and interpretability.

## Extra Question (5 Points)

**Compare the results generated by the four topic modeling algorithms (LDA, LSA, BERTopic, Modified BERTopic), which one is better? You should explain the reasons in details.**

**This question will compensate for any points deducted in this exercise. Maximum marks for the exercise is 100 points.**

In [ ]:
The four topic modeling algorithms (LDA, LSA, BERTopic, and Modified BERTopic) were compared based on topic coherence, interpretability, and semantic understanding.

LDA (Latent Dirichlet Allocation) produces interpretable topics using a probabilistic approach. It performs well for traditional topic modeling but has limitations because it relies on bag-of-words representation and does not capture deep semantic relationships between words.

LSA (Latent Semantic Analysis) uses matrix factorization (SVD) to identify hidden structures in text. It is computationally efficient but produces less interpretable topics and lacks probabilistic meaning, making it weaker compared to LDA.

BERTopic outperforms both LDA and LSA because it uses transformer-based embeddings (BERT), which capture contextual and semantic meaning of text. It produces more coherent and meaningful topics and achieves higher coherence scores.

Modified BERTopic, which uses SentenceTransformer embeddings and DBSCAN clustering, further improves semantic representation. However, DBSCAN is sensitive to parameters and may produce unstable or noisy clusters compared to the default HDBSCAN used in BERTopic.

Overall, BERTopic (default) is the best model because it provides a balance between high coherence, strong semantic understanding, and stable clustering. Modified BERTopic can sometimes perform better in semantic richness but lacks consistency.

# Mandatory Question

**Important: Reflective Feedback on this exercise**

Please provide your thoughts and feedback on the exercises you completed in this assignment.

Consider the following points in your response:

**Learning Experience:** Describe your overall learning experience in working with text data and extracting features using various topic modeling algorithms. Did you understand these algorithms and did the implementations helped in grasping the nuances of feature extraction from text data.

**Challenges Encountered:** Were there specific difficulties in completing this exercise?

Relevance to Your Field of Study: How does this exercise relate to the field of NLP?

**(Your submission will not be graded if this question is left unanswered)**



In [ ]:
Learning Experience

This assignment helped me understand how topic modeling works using LDA, LSA, and BERTopic. I learned how text data is cleaned, converted into numerical form, and how topics are extracted. Implementing these models helped me better understand feature extraction from text data.

Challenges Encountered

Some challenges included understanding coherence score, selecting the optimal number of topics, and handling BERTopic dependencies. Setting up GPT and clustering methods was also slightly difficult.

Relevance to NLP

This exercise is very relevant to NLP because topic modeling is widely used in document classification, text analysis, and information retrieval. It helped me understand real-world applications of NLP techniques.